# Lab 12: Ingestion at scale

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [21]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/prekshapraveen/Documents/lantern-lab2-submission.nosync | LLM mode: replay


Set `ONLINE = True` (and put your team name and Northeastern email in `H`) to call EDGAR for real. Offline, the slide code below reads structure samples from `data/edgar_fixture/` (fictional values).

In [22]:
ONLINE = False
import requests
class _FixtureResponse:
    def __init__(self, path): self.path = path
    def json(self): return json.load(open(self.path))
def _offline_get(url, headers=None, timeout=None):
    if "submissions/CIK" in url: return _FixtureResponse("data/edgar_fixture/submissions_SAMPLE.json")
    if "companyfacts/CIK" in url: return _FixtureResponse("data/edgar_fixture/companyfacts_SAMPLE.json")
    raise RuntimeError(f"offline: no fixture for {url}")
if not ONLINE:
    requests.get = _offline_get

In [23]:
# From the slide: The submissions API: find exactly the filings you need
import requests
H = {"User-Agent": "NEU-DAMG7245 Team 1 praveen.p@northeastern.edu"}

def submissions(cik: str) -> dict:
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    return requests.get(url, headers=H, timeout=30).json()

def earnings_8ks(cik: str, n: int = 4) -> list[dict]:
    r = submissions(cik)["filings"]["recent"]  # columns
    rows = zip(r["form"], r["items"], r["filingDate"],
               r["accessionNumber"], r["primaryDocument"])
    out = [dict(form=f, items=i, date=d, accn=a, doc=p)
           for f, i, d, a, p in rows
           if f == "8-K" and "2.02" in i.split(",")]
    return out[:n]         # newest first

In [24]:
earnings_8ks("0000000001")

[{'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-08-05',
  'accn': '0000000001-26-000041',
  'doc': 'smpa-8k_20260805.htm'},
 {'form': '8-K',
  'items': '2.02,9.01',
  'date': '2026-05-06',
  'accn': '0000000001-26-000022',
  'doc': 'smpa-8k_20260506.htm'}]

In [25]:
# From the slide: companyfacts at scale: facts, not documents
import pandas as pd
KEEP = ("start", "end", "val", "fy", "fp", "form",
        "accn", "frame")

def facts_frame(cik: str) -> pd.DataFrame:
    url = ("https://data.sec.gov/api/xbrl/"
           f"companyfacts/CIK{cik}.json")
    j = requests.get(url, headers=H, timeout=30).json()
    rows = []
    for tax, concepts in j["facts"].items():
        for concept, body in concepts.items():
            for unit, obs in body["units"].items():
                for o in obs:
                    r = {k: o.get(k) for k in KEEP}
                    r.update(cik=cik, taxonomy=tax,
                             concept=concept, unit=unit)
                    rows.append(r)
    return pd.DataFrame(rows)

In [26]:
facts_frame("0000000001")[["concept", "unit", "end", "val", "fp", "form"]]

,concept,unit,end,val,fp,form
0,EntityCommonStockSharesOutstanding,shares,2026-07-31,61000000,Q2,10-Q
1,Revenues,USD,2025-12-31,820000000,FY,10-K
2,Revenues,USD,2026-06-30,430000000,Q2,10-Q
3,NetIncomeLoss,USD,2025-12-31,71000000,FY,10-K


The data room arrives as CS1-style records (`data/dataroom/deal_a/export/records.jsonl`), already in `lantern/2.0`. Value-tiered routing decides which path each source takes:

In [27]:
print(llm.P["ingest"]["routing"])
rec = sd.records("deal_a")[0]
print({k: rec[k] for k in ("schema", "doc_id", "deal_id", "source_type", "tier", "page", "block_id", "citation")})

{'comps': 'xbrl+docling_items', 'dataroom': 'cs1_full+docling+ocr', 'web': 'html_text'}
{'schema': 'lantern/2.0', 'doc_id': 'DR-deal_a-001', 'deal_id': 'deal_a', 'source_type': 'dataroom', 'tier': 4, 'page': 1, 'block_id': 'p0001_b001', 'citation': {'kind': 'pdf_bbox', 'doc_id': 'DR-deal_a-001', 'page': 1, 'bbox': [72.0, 91.5, 528.8, 143.0]}}


In [28]:
# From the slide: The synthetic web: crawl from the sitemap, inside an allowlist
import hashlib, xml.etree.ElementTree as ET
from urllib.parse import urlparse
from bs4 import BeautifulSoup
SM = "{http://www.sitemaps.org/schemas/sitemap/0.9}"

def sitemap_urls(xml_text):
    return [l.text.strip()
            for l in ET.fromstring(xml_text).iter(f"{SM}loc")]

def allowed(url, hosts):            # refuse everything else
    u = urlparse(url)
    return u.scheme == "https" and u.hostname in hosts

def web_record(url, html, deal_id):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style", "nav"]):
        t.decompose()
    text = " ".join(soup.get_text(" ").split())
    sha = hashlib.sha256(html.encode()).hexdigest()
    return {"schema": "lantern/2.0", "doc_id": f"WEB-{sha[:8]}",
            "deal_id": deal_id, "source_type": "synthetic_web",
            "tier": 5, "text": text, "source_path": url,
            "citation": {"kind": "web_span", "url": url},
            "lineage": {"parents": [], "source_sha256": sha},
            "ingest_flags": []}

In [29]:
# Offline, URLs on the course host map to files in data/web_site/
from urllib import robotparser
HOSTS = {llm.P["web"]["host"]}
urls = sitemap_urls(open("data/web_site/sitemap.xml").read())
rp = robotparser.RobotFileParser(); rp.parse(open("data/web_site/robots.txt").read().splitlines())
web = []
for u in urls:
    if not allowed(u, HOSTS) or not rp.can_fetch("LANTERN", u):
        print("refused:", u); continue
    html = open("data/web_site/" + urlparse(u).path.lstrip("/")).read()
    web.append(web_record(u, html, "deal_a"))
print(len(web), "web records;", web[0]["doc_id"], "tier", web[0]["tier"], "|", web[0]["text"][:60])

refused: https://not-allowed.example.com/tracker.html
3 web records; WEB-ed15cb2c tier 5 | About Wren About Wren Freight Software Wren builds dispatch 


**Exercise 12.** Point `earnings_8ks` at your CS1 company with `ONLINE = True`. How many 8-K filings report Item 2.02 in the last year?

In [30]:
# Exercise 12: live EDGAR for our CS1 company. The cells above stay offline on the kit's fixture.
import importlib, requests
importlib.reload(requests)   # undo In [2]'s offline patch of requests.get, for this cell onward
H = {"User-Agent": "NEU-DAMG7245 <team name> <your>@northeastern.edu"}

import pandas as pd
from datetime import date, timedelta
cik = "0000320193"   # AAPL, our CS1 company
cut = (date.today() - timedelta(days=365)).isoformat()
hits = [f for f in earnings_8ks(cik, n=50) if f["date"] >= cut]
print(len(hits)); pd.DataFrame(hits)[["date", "items", "accn"]]

4


,date,items,accn
0,2026-07-30,"2.02,9.01",0000320193-26-000018
1,2026-04-30,"2.02,9.01",0000320193-26-000011
2,2026-01-29,"2.02,9.01",0000320193-26-000005
3,2025-10-30,"2.02,9.01",0000320193-25-000077


### Exercise 12: answer

**How many?** For Apple (CIK 0000320193) there are **4** Item 2.02 8-Ks in the last year:

| Filed | Accession | Quarter reported |
|---|---|---|
| 2025-10-30 | 0000320193-25-000077 | Q4 FY2025 (quarter ended Sept 27, 2025) |
| 2026-01-29 | 0000320193-26-000005 | Q1 FY2026 (ended Dec 27, 2025) |
| 2026-04-30 | 0000320193-26-000011 | Q2 FY2026 (ended Mar 28, 2026) |
| 2026-07-30 | 0000320193-26-000018 | Q3 FY2026 (ended Jun 27, 2026) |

Four is what we expect: one earnings release per quarter, roughly three months apart. There is no
extra pre-announcement or restated release in this window. Apple's Q4 FY2026 release (for the
quarter ending late September 2026) falls just outside the window, because it is filed in late October.

**What the items show:** every filing reads `2.02,9.01`. Item 2.02 is "Results of Operations and
Financial Condition", i.e. the earnings announcement; Item 9.01 is "Financial Statements and
Exhibits", the exhibit list that carries the press release itself as EX-99.1.

**Apple's fiscal year:** Apple's fiscal year ends on the last Saturday of September, so its fiscal
Q1 is October–December and its fourth-quarter results come out in late October, not in
January/February as for a December year-end company. This matters when comparing companies:
the same fiscal label (e.g. "Q1") covers different calendar months for different filers, so
comparables have to be aligned on actual period start and end dates, not on fiscal labels.